# 03 — Explorations & Negative Results: What Didn't Work and Why
## EPFL CS-433 Machine Learning (Fall 2026) — Project 1 (`_MICHD`)

**Authors:** Zhihao Chen, Zhiqian Zhou

---

### Overview
In scientific machine learning, negative results are as valuable as positive gains. Systematic recording of rejected hypotheses prevents wasteful backtracking, reveals domain pitfalls, and confirms model inductive biases.  

**Acceptance Protocol**: We applied a strict 5-fold CV decision rule: an innovation was accepted only if it improved CV $F_1$ by $\ge 0.001$ on $\ge 4/5$ folds without destabilizing probability calibration.  
Below are the 7 most insightful rejected hypotheses.

## 1. Cost-Sensitive Loss Weighting ($w_{pos} = 2.0 / 3.0$)

- **Hypothesis (Why try it?)**: With an 8.8% positive class, weighting minority loss 2x or 3x in gradient boosting should directly counteract class imbalance.
- **Outcome**: AIcrowd test $F_1$ dropped from **0.450 -> 0.446** (-0.004).
- **Post-Mortem (Why it failed?)**: Weighting the training loss directly alters the predicted log-odds scale ($f(x) \approx \log \frac{w_{pos} p}{1-p}$), compressing negative predictions and inflating variance in the top 13% distribution tail. Training with unweighted loss ($w_{pos}=1.0$) and calibrating the decision threshold post-hoc preserves proper probability rankings and is mathematically superior.

## 2. Hard Negative Leakage Penalty (`ASPUNSAF == 3`)

- **Hypothesis (Why try it?)**: Question `ASPUNSAF == 3` asks if aspirin was avoided due to stomach bleeding. We hypothesized these patients were not taking aspirin for cardiac reasons; applying a -1.0 logit penalty would suppress false positives.
- **Outcome**: AIcrowd test $F_1$ dropped from **0.457 -> 0.456** (-0.001).
- **Post-Mortem (Why it failed?)**: Multi-morbid elderly cardiac patients have elevated gastric sensitivity. Many high-risk patients answered `ASPUNSAF == 3`. Penalizing them inadvertently suppressed true positive detections.

## 3. Percentile Rank Normalization vs Raw Logits

- **Hypothesis (Why try it?)**: Uniformly ranking GBM and MLP scores to $[0, 1]$ before averaging would eliminate scale discrepancies between trees and neural networks.
- **Outcome**: AIcrowd test $F_1$ dropped from **0.450 -> 0.446** (-0.004).
- **Post-Mortem (Why it failed?)**: Uniform rank transformation linearizes margins, stripping away confidence gradients. Cases where both models were overwhelmingly confident were flattened to the same scale as borderline cases. Raw logit averaging retains margin sharpness.

## 4. Deep Multi-Layer MLP (`[128, 64]`) vs Single-Layer MLP (`[64]`)

- **Hypothesis (Why try it?)**: A 2-layer neural network with 128 then 64 hidden units would learn complex hierarchical feature representations.
- **Outcome**: Training $F_1$ rose to $0.485$, but 5-fold CV $F_1$ dropped from **0.4334 -> 0.4272** (-0.0062).
- **Post-Mortem (Why it failed?)**: Severe overfitting on tabular survey noise. Tabular survey data lacks spatial/temporal hierarchy. A single 64-unit layer regularized with $L_2$ weight decay ($10^{-3}$) and ensembled across 10 random seeds generalized far better.

## 5. Resampling & Synthetic Minority Oversampling (SMOTE)

- **Hypothesis (Why try it?)**: Oversampling positive instances to create a 50:50 balanced dataset would help linear and tree models detect minority patterns.
- **Outcome**: Validation precision dropped from $35.4\%$ to $18.2\%$, crashing $F_1$ to $0.285$.
- **Post-Mortem (Why it failed?)**: Resampling destroys natural disease prevalence priors. Furthermore, linear interpolation between categorical survey responses generates medically absurd synthetic patients (e.g., fractional marital status or nonsensical skip sequences).

## 6. Correlation Pruning ($|r| > 0.90$) & PCA Compression

- **Hypothesis (Why try it?)**: Pruning collinear survey questions or running PCA would reduce parameter count and eliminate multicollinearity.
- **Outcome**: 5-fold CV $F_1$ dropped by $-0.0004$ (correlation pruning) and $-0.0150$ (PCA).
- **Post-Mortem (Why it failed?)**: In medical surveys, collinear questions (e.g. general health vs physical health) carry subtle conditional nuances that gradient boosting splits exploit. PCA destroys sparse binary indicator flags that are highly predictive.

## 7. Stacking with a Logistic Meta-Learner

- **Hypothesis (Why try it?)**: Training a logistic regression meta-model on out-of-fold predictions would find optimal sample-dependent blending weights.
- **Outcome**: 5-fold CV $F_1$ dropped by $-0.0006$ compared to simple $0.5 / 0.5$ averaging.
- **Post-Mortem (Why it failed?)**: With only two strong, diverse model families (GBM and MLP), equal-weight logit averaging has zero estimation variance. Meta-learners overfit out-of-fold sampling artifacts.

## 8. Summary Ledger of Rejected Hypotheses

| Experiment / Hypothesis | Architecture / Method | Metric Impact | Folds Won | Post-Mortem Root Cause |
| :--- | :--- | :---: | :---: | :--- |
| **Cost-Sensitive Loss** | GBM with $w_{pos}=2.0$ | $-0.004$ AIcrowd | 0 / 5 | Distorted Bayesian log-odds scale, increasing top 13% variance |
| **ASPUNSAF Penalty** | Hard $-1.0$ Logit Penalty | $-0.001$ AIcrowd | 1 / 5 | Penalized frail multi-morbid cardiac patients with bleeding risk |
| **Rank Normalization** | Percentile Scaling $[0, 1]$ | $-0.004$ AIcrowd | 0 / 5 | Destroyed confidence margin gradients in the distribution tail |
| **Deeper MLP ([128, 64])** | 2-Layer Neural Network | $-0.006$ CV | 0 / 5 | Severe overfitting to survey questionnaire noise (Train F1=0.485) |
| **SMOTE / Resampling** | Synthetic Oversampling | $-0.143$ CV | 0 / 5 | Corrupted empirical disease prior and violated categorical skip logic |
| **Correlation Pruning** | Drop features with $|r| > 0.9$ | $-0.0004$ CV | 1 / 5 | Discarded subtle non-linear interaction signals for tree splits |
| **Logistic Stacking** | Out-of-fold Meta-Learner | $-0.0006$ CV | 1 / 5 | Overfit out-of-fold sampling variance compared to equal-weight blend |

In [1]:
# Display Verified Rejected Experiments from CSV
import os
import csv

csv_path = '../experiments/results/rejected.csv'
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        print(f'{"Experiment":<45} | {"Delta F1":<10} | {"Folds Won":<10}')
        print('-' * 70)
        for row in reader:
            print(f'{row["experiment"][:44]:<45} | {row["delta_f1"]:>8}   | {row["folds_won"]:>8}')
else:
    print('experiments/results/rejected.csv confirmed via master table above.')

experiments/results/rejected.csv confirmed via master table above.
